In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)
print('Raw dir exists:', raw_dir.exists())
print('Processed dir exists:', processed_dir.exists())


In [ ]:
# Charger les jeux de données bruts
train_raw = pd.read_csv(raw_dir / 'train.csv', low_memory=False)
store_raw = pd.read_csv(raw_dir / 'store.csv', low_memory=False)

print('train_raw shape:', train_raw.shape)
print('store_raw shape:', store_raw.shape)
print('\nstate holiday before:', sorted(train_raw['StateHoliday'].astype(str).unique().tolist()))


In [ ]:
# Normaliser les dates et les valeurs
train_clean = train_raw.copy()
train_clean['Date'] = pd.to_datetime(train_clean['Date'])
train_clean['StateHoliday'] = train_clean['StateHoliday'].astype(str).str.strip().str.lower()
train_clean['StateHoliday'] = train_clean['StateHoliday'].replace({'0': '0', 'nan': '0', 'none': '0', 'na': '0'})

# Règle métier: les jours fermés ne sont pas utilisés pour l'entraînement
train_clean['is_closed'] = train_clean['Open'] == 0
train_clean = train_clean[train_clean['Open'] == 1].copy()
train_clean = train_clean[train_clean['Sales'] > 0].copy()

# Clean store metadata
store_clean = store_raw.copy()
store_clean['CompetitionDistance_missing'] = store_clean['CompetitionDistance'].isna()
store_clean['CompetitionDistance'] = store_clean['CompetitionDistance'].fillna(store_clean['CompetitionDistance'].median())
for col in ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']:
    store_clean[f'{col}_missing'] = store_clean[col].isna()
    store_clean[col] = store_clean[col].fillna(-1)
for col in ['Promo2SinceWeek', 'Promo2SinceYear']:
    store_clean[f'{col}_missing'] = store_clean[col].isna()
    store_clean[col] = store_clean[col].fillna(-1)
store_clean['PromoInterval'] = store_clean['PromoInterval'].fillna('none').astype(str).str.lower()

print('train_clean shape:', train_clean.shape)
print('store_clean shape:', store_clean.shape)
print('\nStateHoliday after:', sorted(train_clean['StateHoliday'].astype(str).unique().tolist()))
print('\nCompetitionDistance missing after fill:', store_clean['CompetitionDistance_missing'].sum())


In [ ]:
# Jointure finale et stockage des données propres
merged = train_clean.merge(store_clean, on='Store', how='left')
print('merged shape:', merged.shape)
print('\nMissing values after merge:\n', merged.isna().sum().to_string())

# Sauvegarder les données traitées
train_clean.to_parquet(processed_dir / 'train_clean.parquet', index=False)
store_clean.to_parquet(processed_dir / 'store_clean.parquet', index=False)
merged.to_parquet(processed_dir / 'train_store_clean.parquet', index=False)

print('\nSaved files:')
for file in ['train_clean.parquet', 'store_clean.parquet', 'train_store_clean.parquet']:
    path = processed_dir / file
    print(file, path.exists(), path.stat().st_size if path.exists() else 0)
